In [1]:
from typing import Callable
from types import SimpleNamespace
from pipe import select
from omegaconf import DictConfig, OmegaConf

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy import linalg

import plotly.express as px
import plotly.graph_objects as go

import warnings
warnings.filterwarnings('ignore', category=np.exceptions.ComplexWarning)

In [ ]:
np.random.seed(123)

In [2]:
# system params
systems = [
    {
        "gamma": 1.,
        "omega": -0.9,
        "u": np.array([1., 0]),
        "v": np.array([0., 1.]),
    },
    {
        "gamma": 0.5,
        "omega": -1.,
        "u": np.array([1., 0]),
        "v": np.array([0., 1.]),
    }
]
systems = list(systems | select(lambda d: SimpleNamespace(**d)))

In [3]:
# building matrix A in our ODE x' = A @ x
def system_matrix(
    sys_c: SimpleNamespace
):
    L = np.diag(
        [-sys_c.gamma + 1j*sys_c.omega, -sys_c.gamma - 1j*sys_c.omega]
    )
    e1, e2 = list(
        [sys_c.u + 1j*sys_c.v, sys_c.u - 1j*sys_c.v] |
        select(lambda e: e / linalg.norm(e)) |
        select(lambda e: e[:, None])
    )
    U = np.hstack([e1, e2])
    A = (U @ L @ U.T.conj()).astype(float)
    return A

In [4]:
# exact solution
def solution(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    num_points = t.size
    # amplitude
    ampl = linalg.norm(x0)
    if x0[0] > 0:
        phase = np.arctan(x0[1] / x0[0])
    elif x0[0] < 0:
        phase = np.pi + np.arctan(x0[1] / x0[0])
    else:
        phase = np.sign(x0[1]) * np.pi / 2

    u_rep, v_rep = list(
        [sys_c.u, sys_c.v] |
        select(lambda x: np.repeat(x[:, None], num_points, axis=1))
    )    
    sol = ampl * np.exp(-sys_c.gamma * t) * \
        (np.cos(phase - sys_c.omega * t) * u_rep + np.sin(phase - sys_c.omega * t) * v_rep)
    return sol.T

In [5]:
# system's matrices
sys_matr = list(systems | select(lambda d: system_matrix(d)))
sys_matr

[array([[-1. , -0.9],
        [ 0.9, -1. ]]),
 array([[-0.5, -1. ],
        [ 1. , -0.5]])]

In [6]:
x0 = np.array([1., 1.])

In [7]:
T = 1.
delta_t = 0.5e-1
t = np.linspace(0., T, int(T / delta_t))
print(t.size)

20


## Solutions without jumps

In [8]:
# compute exact solutions
true_sol = list(
    systems |
    select(lambda d: solution(t, d, x0))
)

In [9]:
def RK_solver_f(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    A = system_matrix(sys_c)
    return solve_ivp(
        lambda t, x: A.dot(x),
        [t[0], t[-1]],
        x0,
        method="RK23",
        t_eval=t
    ).y.T

In [10]:
# solve sytems with RK23
RK23_sol = list(
    systems |
    select(lambda s: RK_solver_f(t, s, x0))
)

In [11]:
RK23_sol[0].shape

(20, 2)

In [12]:
solutions = {
    "true": true_sol,
    "RK23": RK23_sol
}

In [13]:
fig = go.Figure()
for sol_name, sol in solutions.items():
    for i, sys_sol in enumerate(sol):
        line = dict(dash="dash") if sol_name.count("true") == 0 else None
        fig.add_trace(
            go.Scatter(
                x=sys_sol[:, 0],
                y=sys_sol[:, 1],
                mode='lines',
                name=f"{sol_name}_{i}",
                line=line
            )
        )
fig.add_trace(
    go.Scatter(
        x=[x0[0]],
        y=[x0[1]],
        name="start"
    )
)
fig.show()

## Introduce jumps to the model

Jumps are shocks which suddenly increase the speed

In [14]:
jump_ampl = 0.09
num_jumps = 3
jump_indx = np.arange(1, num_jumps + 1) * (len(t) // (num_jumps + 2))
t[jump_indx]

array([0.21052632, 0.42105263, 0.63157895])

In [15]:
def jumped_solution(
    t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray, 
    jump_indxs: np.ndarray, jump_ampl: float
):
    t = t.copy()
    jump_indxs = jump_indxs

    cur_x0 = x0
    cur_start = 0
    sol = []
    for jump_indx in jump_indxs:
        cur_t = t[cur_start:jump_indx + 1] - t[cur_start]
        cur_sol = solution(cur_t, sys_c, cur_x0)

        sol.append(cur_sol[:-1])
        # recompute jumped x0
        cur_x0 = cur_sol[-1].copy()
        cur_x0[1] += jump_ampl
        # adjust time for next computation
        cur_start = jump_indx
    
    # append solution for the last segment
    cur_t = t[cur_start:] - t[cur_start]
    sol.append(solution(cur_t, sys_c, cur_x0))

    return np.concat(sol)

In [16]:
# compute exact solution for system 0
jumped_solution_0 = jumped_solution(t, systems[0], x0, jump_indx, jump_ampl)

In [17]:
jumped_solution_0.shape

(20, 2)

In [18]:
fig = go.Figure()
for sol_name, sol in solutions.items():
    for i, sys_sol in enumerate(sol):
        line = dict(dash="dash") if sol_name.count("true") == 0 else None
        fig.add_trace(
            go.Scatter(
                x=sys_sol[:, 0],
                y=sys_sol[:, 1],
                mode='lines',
                name=f"{sol_name}_{i}",
                line=line
            )
        )
fig.add_trace(
    go.Scatter(
        x=[x0[0]],
        y=[x0[1]],
        name="start"
    )
)
fig.add_trace(
    go.Scatter(
        x=jumped_solution_0[:, 0],
        y=jumped_solution_0[:, 1],
        mode='lines',
        name="jumped_0"
    )
)
fig.show()

## Perform classification with jumps

In [33]:
NUM_SAMPLES = 1000
sigma = 0.3
# what system to classify
num_traj_points = jumped_solution_0.shape[0]

In [34]:
def cls_loss(sample_traj: np.ndarray, mean_traj: np.ndarray):
    return np.linalg.norm(sample_traj - mean_traj, ord=2, axis=1).mean()

In [35]:
accuracy_df = []
cur_accuracy_df = pd.DataFrame(
    {
        "accuracy": np.zeros([len(solutions)]),
        "method": ["true", "RK23"],
        "sigma": np.full([len(solutions)], sigma)
    }
)
for i in range(NUM_SAMPLES):
    sample_traj = jumped_solution_0 + sigma * np.random.randn(num_traj_points, 2)
    for sol_name, sol in solutions.items():
        sol_losses = list(
            range(2) |
            select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
        )
        if sol_losses[0] < sol_losses[1]:
            cur_accuracy_df.loc[
                cur_accuracy_df["method"] == sol_name, 
                "accuracy"
            ] += 1.
cur_accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES
accuracy_df.append(cur_accuracy_df)

In [36]:
pd.concat(accuracy_df, axis=0)

,accuracy,method,sigma
0,0.0,true,0.3
1,0.0,RK23,0.3


## Perfrom classification with trajectory segmenting

In [37]:
NUM_SAMPLES = 500

In [38]:
segment_sizes = np.array([3, 5, 10])

In [39]:
def segmented_solution(
    t: np.ndarray, sys_c: SimpleNamespace, traj: np.ndarray, 
    segment_size: int, solver_f: Callable
):
    sol = []
    cur_start = 0
    while cur_start < t.size:
        cur_x0 = traj[cur_start]
        sol.append(
            solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                cur_x0
            )
        )
        cur_start = cur_start + segment_size

    return np.concat(sol)

In [40]:
accuracy_df = []
for seg_size in segment_sizes:
    cur_accuracy_df = pd.DataFrame(
        {
            "accuracy": np.zeros([len(solutions)]),
            "method": [f"true_{seg_size}", f"RK23_{seg_size}"],
            "sigma": np.full([len(solutions)], sigma)
        }
    )
    for i in range(NUM_SAMPLES):
        sample_traj = jumped_solution_0 + sigma * np.random.randn(num_traj_points, 2)

        true_seg_solutions = list(
            systems |
            select(lambda s: segmented_solution(t, s, sample_traj, seg_size, solution))
        )
        rk_seg_solutions = list(
            systems |
            select(lambda s: segmented_solution(t, s, sample_traj, seg_size, RK_solver_f))
        )

        if i % 250 == 0:
            print("Segment size =", seg_size)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=jumped_solution_0[:, 0],
                    y=jumped_solution_0[:, 1],
                    mode='lines',
                    name="jumped_0"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=sample_traj[:, 0],
                    y=sample_traj[:, 1],
                    mode='markers',
                    name="jumped_sample"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=true_seg_solutions[0][:, 0],
                    y=true_seg_solutions[0][:, 1],
                    mode='lines',
                    name="segment0_true"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=rk_seg_solutions[0][:, 0],
                    y=rk_seg_solutions[0][:, 1],
                    mode='lines',
                    name="segment0_rk",
                    line=dict(dash="dash")
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=true_seg_solutions[1][:, 0],
                    y=true_seg_solutions[1][:, 1],
                    mode='lines',
                    name="segment1_true"
                )
            )
            fig.show()

        solutions = {
            f"true_{seg_size}": true_seg_solutions,
            f"RK23_{seg_size}": rk_seg_solutions
        }

        for sol_name, sol in solutions.items():
            sol_losses = list(
                range(2) |
                select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
            )
            if sol_losses[0] < sol_losses[1]:
                cur_accuracy_df.loc[
                    cur_accuracy_df["method"] == sol_name, 
                    "accuracy"
                ] += 1.
    cur_accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES
    accuracy_df.append(cur_accuracy_df)

Segment size = 3


Segment size = 3


Segment size = 5


Segment size = 5


Segment size = 10


Segment size = 10


In [41]:
accuracy_df = pd.concat(accuracy_df, axis=0)
accuracy_df

,accuracy,method,sigma
0,0.838,true_3,0.3
1,0.838,RK23_3,0.3
0,0.770,true_5,0.3
1,0.770,RK23_5,0.3
0,0.648,true_10,0.3
1,0.650,RK23_10,0.3


## Perfrom classification with trajectory segmenting when sigma is high

In [42]:
segment_sizes = np.array([5, 10])

In [43]:
fig = go.Figure()
sample_traj = jumped_solution_0 + sigma * np.random.randn(num_traj_points, 2)
fig.add_trace(
    go.Scatter(
        x=jumped_solution_0[:, 0],
        y=jumped_solution_0[:, 1],
        mode='lines',
        name="jumped_0"
    )
)
fig.add_trace(
    go.Scatter(
        x=sample_traj[:, 0],
        y=sample_traj[:, 1],
        mode='markers',
        name="jumped_sample"
    )
)
fig.show()

In [44]:
from scipy import optimize

def segmented_solution_adv(
    t: np.ndarray, sys_c: SimpleNamespace, traj: np.ndarray, 
    segment_size: int, solver_f: Callable
):
    sol = []
    cur_start = 0
    while cur_start < t.size:
        # initial for true x0
        cur_x0 = traj[cur_start]
        def obj(x0: np.ndarray):
            pred = solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                x0
            )
            true = traj[cur_start: min(cur_start + segment_size, t.size)]
            return cls_loss(true, pred)
        cur_x0 = optimize.minimize(obj, cur_x0, method="BFGS", options=dict(maxiter=200)).x

        sol.append(
            solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                cur_x0
            )
        )
        cur_start = cur_start + segment_size

    return np.concat(sol)

In [45]:
accuracy_df = []
for seg_size in segment_sizes:
    cur_accuracy_df = pd.DataFrame(
        {
            "accuracy": np.zeros([len(solutions)]),
            "method": [f"true_{seg_size}", f"RK23_{seg_size}"],
            "sigma": np.full([len(solutions)], sigma)
        }
    )
    for i in range(NUM_SAMPLES):
        sample_traj = jumped_solution_0 + sigma * np.random.randn(num_traj_points, 2)

        true_seg_solutions = list(
            systems |
            select(lambda s: segmented_solution_adv(t, s, sample_traj, seg_size, solution))
        )
        rk_seg_solutions = list(
            systems |
            select(lambda s: segmented_solution_adv(t, s, sample_traj, seg_size, RK_solver_f))
        )

        if i % 250 == 0:
            print("Segment size =", seg_size)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=jumped_solution_0[:, 0],
                    y=jumped_solution_0[:, 1],
                    mode='lines',
                    name="jumped_0"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=sample_traj[:, 0],
                    y=sample_traj[:, 1],
                    mode='markers',
                    name="jumped_sample"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=true_seg_solutions[0][:, 0],
                    y=true_seg_solutions[0][:, 1],
                    mode='lines',
                    name="segment0_true"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=rk_seg_solutions[0][:, 0],
                    y=rk_seg_solutions[0][:, 1],
                    mode='lines',
                    name="segment0_rk",
                    line=dict(dash="dash")
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=true_seg_solutions[1][:, 0],
                    y=true_seg_solutions[1][:, 1],
                    mode='lines',
                    name="segment1_true"
                )
            )
            fig.show()

        solutions = {
            f"true_{seg_size}": true_seg_solutions,
            f"RK23_{seg_size}": rk_seg_solutions
        }

        for sol_name, sol in solutions.items():
            sol_losses = list(
                range(2) |
                select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
            )
            if sol_losses[0] < sol_losses[1]:
                cur_accuracy_df.loc[
                    cur_accuracy_df["method"] == sol_name, 
                    "accuracy"
                ] += 1.
    cur_accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES
    accuracy_df.append(cur_accuracy_df)

Segment size = 5


Segment size = 5


Segment size = 10


Segment size = 10


In [46]:
accuracy_df = pd.concat(accuracy_df, axis=0)
accuracy_df

,accuracy,method,sigma
0,0.450,true_5,0.3
1,0.450,RK23_5,0.3
0,0.482,true_10,0.3
1,0.484,RK23_10,0.3
